# 02 — Silver Layer

**Tujuan:** Membersihkan data Bronze Layer, UNNEST kolom `nutriments`, encoding, dan feature engineering.

**Input:** `/home/jovyan/work/data/bronze/food_raw` (Delta Lake)  
**Output:** `/home/jovyan/work/data/silver/food_clean` (Delta Lake, partisi by `nova_group`)

**Pipeline:**
1. Setup SparkSession
2. Baca Bronze Layer
3. Filter `nova_group IS NOT NULL`
4. UNNEST `nutriments` → kolom flat
5. Filter outlier + handle missing values
6. Encoding + normalisasi + feature engineering
7. Simpan ke Delta Lake
8. Validasi hasil

## 1. Setup SparkSession

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[2]") \
    .appName("02-silver-layer") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.driver.memory", "2g") \
    .config("spark.driver.maxResultSize", "1g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Mode:", spark.sparkContext.master)

Spark version: 3.5.1
Mode: local[2]


## 2. Baca Bronze Layer

In [2]:
df_bronze = spark.read.format("delta").load("/home/jovyan/work/data/bronze/food_raw")

print("Jumlah baris:", df_bronze.count())
print("Jumlah kolom:", len(df_bronze.columns))

Jumlah baris: 4487169
Jumlah kolom: 112


## 3. Filter `nova_group IS NOT NULL`

In [3]:
df_filtered = df_bronze.filter(df_bronze.nova_group.isNotNull())

print("Baris setelah filter nova_group IS NOT NULL:", df_filtered.count())
print("Baris yang di-drop:", 4487169 - df_filtered.count())

Baris setelah filter nova_group IS NOT NULL: 1119410
Baris yang di-drop: 3367759


## 4. UNNEST `nutriments` → Kolom Flat

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

df_pivot = df_filtered.select(
    col("*"),
    *[
        F.expr(f"""
            aggregate(
                filter(nutriments, x -> x.name = '{n}'),
                cast(null as float),
                (acc, x) -> x.`100g`
            )
        """).alias(f"{n.replace('-', '_')}_100g")
        for n in ['energy-kcal', 'fat', 'sugars', 'proteins', 'salt']
    ]
).drop("nutriments")

print("Kolom flat tersedia:")
for c in ['energy_kcal_100g', 'fat_100g', 'sugars_100g', 'proteins_100g', 'salt_100g']:
    print(f"  - {c}")
print("Total kolom:", len(df_pivot.columns))

Kolom flat tersedia:
  - energy_kcal_100g
  - fat_100g
  - sugars_100g
  - proteins_100g
  - salt_100g
Total kolom: 116


## 5. Filter Outlier + Handle Missing Values (Median)

In [5]:
from pyspark.sql.functions import when, lit

nutrisi_cols = ['energy_kcal_100g', 'fat_100g', 'sugars_100g', 'proteins_100g', 'salt_100g']

# Filter outlier: nilai harus BETWEEN 0 AND 1000
df_clean = df_pivot
for c in nutrisi_cols:
    df_clean = df_clean.withColumn(c,
        when((col(c) >= 0) & (col(c) <= 1000), col(c)).otherwise(None)
    )

# Isi missing values dengan median
medians = {}
for c in nutrisi_cols:
    median_val = df_clean.approxQuantile(c, [0.5], 0.01)[0]
    medians[c] = median_val
    df_clean = df_clean.withColumn(c,
        when(col(c).isNull(), lit(median_val)).otherwise(col(c))
    )

print("Median per kolom nutrisi:")
for c, v in medians.items():
    print(f"  {c}: {v:.2f}")

Median per kolom nutrisi:
  energy_kcal_100g: 256.00
  fat_100g: 6.44
  sugars_100g: 4.40
  proteins_100g: 5.10
  salt_100g: 0.40


## 6. Encoding + Normalisasi + Feature Engineering

In [6]:
from pyspark.sql.functions import lower, size, array_contains

# Encode nutriscore_grade: A->1, B->2, C->3, D->4, E->5
df_clean = df_clean.withColumn("nutriscore_encoded",
    when(col("nutriscore_grade") == "a", 1)
    .when(col("nutriscore_grade") == "b", 2)
    .when(col("nutriscore_grade") == "c", 3)
    .when(col("nutriscore_grade") == "d", 4)
    .when(col("nutriscore_grade") == "e", 5)
    .otherwise(None)
)

# Normalisasi teks
df_clean = df_clean.withColumn("brands", lower(col("brands")))

# Handle missing values kolom teks
df_clean = df_clean \
    .withColumn("brands", when(col("brands").isNull(), lit("unknown")).otherwise(col("brands"))) \
    .withColumn("categories", when(col("categories").isNull(), lit("unknown")).otherwise(col("categories")))

# Feature engineering
df_clean = df_clean.withColumn("additives_count",
    when(col("additives_tags").isNull(), 0).otherwise(size(col("additives_tags")))
)
df_clean = df_clean.withColumn("has_palm_oil",
    when(array_contains(col("ingredients_analysis_tags"), "en:palm-oil"), 1).otherwise(0)
)
df_clean = df_clean.withColumn("is_vegan",
    when(array_contains(col("ingredients_analysis_tags"), "en:vegan"), 1).otherwise(0)
)
df_clean = df_clean.withColumn("is_vegetarian",
    when(array_contains(col("ingredients_analysis_tags"), "en:vegetarian"), 1).otherwise(0)
)

print("Feature engineering selesai:")
print("  - nutriscore_encoded (A=1, B=2, C=3, D=4, E=5)")
print("  - brands (lowercase)")
print("  - categories (missing -> 'unknown')")
print("  - additives_count")
print("  - has_palm_oil")
print("  - is_vegan")
print("  - is_vegetarian")
print("Total kolom:", len(df_clean.columns))

Feature engineering selesai:
  - nutriscore_encoded (A=1, B=2, C=3, D=4, E=5)
  - brands (lowercase)
  - categories (missing -> 'unknown')
  - additives_count
  - has_palm_oil
  - is_vegan
  - is_vegetarian
Total kolom: 121


## 7. Simpan ke Delta Lake

In [7]:
import time

start = time.time()

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("nova_group") \
    .option("maxRecordsPerFile", 250000) \
    .save("/home/jovyan/work/data/silver/food_clean")

elapsed = time.time() - start
throughput = 1119410 / elapsed
print(f"Silver Layer tersimpan. Waktu: {elapsed:.1f} detik")
print(f"Throughput Silver: {throughput:,.0f} baris/detik")

Silver Layer tersimpan. Waktu: 189.9 detik
Throughput Silver: 5,895 baris/detik


## 8. Validasi Hasil

In [8]:
from pyspark.sql.functions import col

df_silver = spark.read.format("delta").load("/home/jovyan/work/data/silver/food_clean")

print("Jumlah baris:", df_silver.count())
print("Jumlah kolom:", len(df_silver.columns))
print("Partisi nova_group:", df_silver.select("nova_group").distinct().orderBy("nova_group").collect())

print("\nNull check kolom fitur utama:")
for c in ['energy_kcal_100g', 'fat_100g', 'sugars_100g', 'proteins_100g', 'salt_100g', 'additives_count']:
    null_count = df_silver.filter(col(c).isNull()).count()
    print(f"  {c}: {null_count} null")

Jumlah baris: 1119410
Jumlah kolom: 121
Partisi nova_group: [Row(nova_group=1), Row(nova_group=2), Row(nova_group=3), Row(nova_group=4)]

Null check kolom fitur utama:
  energy_kcal_100g: 0 null
  fat_100g: 0 null
  sugars_100g: 0 null
  proteins_100g: 0 null
  salt_100g: 0 null
  additives_count: 0 null
